# Étape 7 — Matching profil ↔ offre

Pour chaque paire (profil, offre), un **score de 0 à 100** et une **explication** : compétences communes, proches et manquantes.

**Tourne sur CPU** : pas besoin de GPU (Exécution → Modifier le type d'exécution → Processeur).

Déroulé :
1. Charger les profils et les offres.
2. **Créer la fiche de notation** et noter les offres **avant** de voir les scores.
3. Calculer les scores (dictionnaire de normalisation + embeddings multilingues).
4. Évaluer sur tes notes : NDCG@5 et Spearman, avec poids réglés par validation croisée.
5. Refaire l'évaluation avec les compétences **extraites par le modèle v3** au lieu des étiquettes manuelles.

In [ ]:
# 1. Installation + Drive
!pip install -q sentence-transformers pandas scipy

from google.colab import drive
drive.mount('/content/drive')

import os, json, re, random, itertools
import numpy as np
import pandas as pd

RACINE = '/content/drive/MyDrive/job-offer-extractor'
DOSSIER = f'{RACINE}/data'
RESULTATS = f'{RACINE}/resultats'
MATCHING = f'{RACINE}/matching'
os.makedirs(MATCHING, exist_ok=True)
PROFILS = f'{MATCHING}/profils.json'
assert os.path.exists(PROFILS), f'Place profils.json dans {MATCHING}'

In [ ]:
# 2. Charger les profils et les offres (étiquettes manuelles)
def lire(chemin):
    with open(chemin, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

profils = json.load(open(PROFILS, encoding='utf-8'))

offres = {}
for o in lire(f'{DOSSIER}/offres_propres.jsonl'):
    offres[o['url']] = {**o, 'jeu': 'dataset'}
for nom, jeu in [('test.jsonl', 'test'), ('test_vierge.jsonl', 'test_vierge')]:
    chemin = f'{DOSSIER}/{nom}'
    if os.path.exists(chemin):
        for o in lire(chemin):
            offres[o['url']] = {**offres.get(o['url'], {}), **o, 'jeu': jeu}

print(len(profils), 'profils :', [p['id'] for p in profils])
print(len(offres), 'offres :', pd.Series([o['jeu'] for o in offres.values()]).value_counts().to_dict())

## Dictionnaire de normalisation

Il ramène chaque variante à **un seul nom canonique**, des deux côtés (profils et offres), en suivant `GUIDE_ANNOTATION.md`. Les compétences composées sont séparées (« Git/GitHub » → Git, GitHub), sauf CI/CD, TCP/IP, PL/SQL, Pub/Sub.

In [ ]:
# 3. Normalisation des compétences
NORMALISATION = {
    # langages et frameworks
    'js': 'JavaScript', 'javascript': 'JavaScript', 'es6': 'JavaScript', 'جافا سكريبت': 'JavaScript',
    'ts': 'TypeScript', 'typescript': 'TypeScript',
    'python3': 'Python', 'python': 'Python', 'بايثون': 'Python',
    'react.js': 'React', 'reactjs': 'React', 'react js': 'React',
    'vue': 'Vue.js', 'vuejs': 'Vue.js', 'vue 3': 'Vue.js', 'vue3': 'Vue.js',
    'nuxt.js': 'Nuxt', 'nuxtjs': 'Nuxt',
    'node': 'Node.js', 'nodejs': 'Node.js', 'node js': 'Node.js',
    'nest': 'NestJS', 'nest.js': 'NestJS', 'nestjs': 'NestJS',
    'next': 'Next.js', 'nextjs': 'Next.js',
    'express': 'Express.js', 'expressjs': 'Express.js',
    'angularjs': 'Angular', 'angular js': 'Angular',
    'spring': 'Spring', 'springboot': 'Spring Boot',
    'laravel 12': 'Laravel', 'laravel 11': 'Laravel',
    'java 17': 'Java', 'java 21': 'Java', 'java ee': 'JEE',
    'dotnet': '.NET', '.net core': '.NET', 'asp.net core': 'ASP.NET Core',
    'html': 'HTML', 'html5': 'HTML', 'css': 'CSS', 'css3': 'CSS',
    'tailwind': 'Tailwind CSS', 'tailwindcss': 'Tailwind CSS',
    # données
    'postgres': 'PostgreSQL', 'postgresql': 'PostgreSQL', 'postgre': 'PostgreSQL',
    'mongo': 'MongoDB', 'mongodb': 'MongoDB',
    'microsoft sql server': 'SQL Server', 'mssql': 'SQL Server', 'ms sql': 'SQL Server',
    'prisma orm': 'Prisma', 'typeorm orm': 'TypeORM', 'eloquent orm': 'Eloquent',
    'vector databases': 'bases vectorielles', 'vector database': 'bases vectorielles', 'moteurs vectoriels': 'bases vectorielles',
    'modélisation de données': 'data modeling', 'data modelling': 'data modeling', 'database design': 'data modeling',
    'data warehousing': 'data warehouse',
    # cloud / devops
    'microsoft azure': 'Azure', 'azure cloud': 'Azure', 'google cloud platform': 'GCP', 'google cloud': 'GCP',
    'amazon web services': 'AWS', 'lambda': 'AWS Lambda', 'aws lambda': 'AWS Lambda',
    'azuredevops': 'Azure DevOps', 'k8s': 'Kubernetes', 'kubernetes (eks)': 'Kubernetes',
    'github action': 'GitHub Actions', 'gitlab-ci': 'GitLab CI', 'gitlab ci/cd': 'GitLab CI',
    'elk': 'ELK Stack', 'elk stack': 'ELK Stack', 'micro-services': 'microservices', 'microservice': 'microservices',
    'continuous integration': 'intégration continue',
    # API
    'rest api': 'API REST', 'rest apis': 'API REST', 'restful api': 'API REST', 'restful apis': 'API REST',
    'restful services': 'API REST', 'apis rest': 'API REST', 'api restful': 'API REST', 'web apis': 'API REST',
    'webhook': 'webhooks',
    # IA
    'ia': 'intelligence artificielle', 'ai': 'intelligence artificielle', 'artificial intelligence': 'intelligence artificielle',
    'الذكاء الاصطناعي': 'intelligence artificielle',
    'ia générative': 'intelligence artificielle générative', 'generative ai': 'intelligence artificielle générative',
    'ia agentique': 'intelligence artificielle agentique', 'agentic ai': 'intelligence artificielle agentique',
    'ai agents': 'intelligence artificielle agentique',
    'ml': 'Machine Learning', 'apprentissage automatique': 'Machine Learning', 'machine learning': 'Machine Learning',
    'deep learning': 'Deep Learning', 'apprentissage profond': 'Deep Learning',
    'traitement automatique du langage naturel': 'NLP', 'large language models': 'LLM', 'llms': 'LLM',
    # méthodes et domaines
    'oop': 'programmation orientée objet', 'poo': 'programmation orientée objet',
    'rgpd': 'GDPR', 'software testing methodologies': 'Software Testing',
    'information security': "sécurité de l'information", 'أمن المعلومات': "sécurité de l'information",
    'cybersecurity': 'cybersécurité', 'cyber security': 'cybersécurité',
    'responsive': 'responsive design', 'التصميم المتجاوب': 'responsive design',
    'ui/ux': 'UI design', 'ux/ui': 'UI design',
    'تطوير البرمجيات': 'développement logiciel', 'software development': 'développement logiciel',
    'برمجة مواقع': 'développement web', 'تطوير الويب': 'développement web', 'web development': 'développement web',
    'هندسة البرمجيات': 'génie logiciel', 'software engineering': 'génie logiciel',
    'برمجة أندرويد': 'Android', 'برمجة ios': 'iOS', 'mobile development': 'développement mobile',
    'firewall': 'pare-feu', 'firewalls': 'pare-feu',
}
GARDER_SLASH = {'ci/cd', 'tcp/ip', 'pl/sql', 'pub/sub', 'tests a/b'}

def cle(s):
    s = re.sub(r'\s+', ' ', str(s).strip().lower())
    return s.strip(' ;,').rstrip('.')

def canon(competence):
    """Une compétence brute -> liste de noms canoniques (séparation des composés)."""
    k = cle(competence)
    if not k:
        return []
    morceaux = [k] if k in GARDER_SLASH or k in NORMALISATION else [m.strip() for m in k.split('/') if m.strip()]
    sortie = []
    for m in morceaux:
        m = re.sub(r'\s*\((.*?)\)\s*', ' ', m).strip() if m not in NORMALISATION else m
        sortie.append(NORMALISATION.get(m, m))
    return sortie

def ensemble(competences):
    """Liste brute -> ensemble de clés canoniques (minuscules)."""
    return {cle(c) for x in (competences or []) if isinstance(x, str) for c in canon(x)}

# exemples
for x in ['.NET', 'ASP.NET / .NET', 'React.js', 'Git/GitHub', 'CI/CD', 'Prisma ORM', 'REST APIs', 'بايثون', 'HTML5/CSS3', 'Lambda']:
    print(f'{x:12} -> {canon(x)}')

## Fiche de notation (À FAIRE AVANT DE VOIR LES SCORES)

On note les **28 offres** pour lesquelles on a aussi les sorties du modèle v3 (14 du test + 14 du test vierge), ce qui permettra l'évaluation de bout en bout à l'étape 5.

Pour chaque profil et chaque offre, mets une note :
- **0** = hors sujet
- **1** = faible correspondance
- **2** = intéressante
- **3** = je postulerais / elle postulerait

Note en te mettant à la place de la personne (compétences, type de contrat, lieu). **N'exécute pas les cellules suivantes avant d'avoir noté.**

In [ ]:
# 4. Créer la fiche de notation
NOTATION = f'{MATCHING}/notation.csv'
NOTATION_REMPLIE = f'{MATCHING}/notation_remplie.csv'

a_noter = [o for o in offres.values() if o['jeu'] in ('test', 'test_vierge')]
random.Random(7).shuffle(a_noter)

lignes = []
for o in a_noter:
    e = o['extraction']
    ligne = {'url': o['url'], 'poste': e['poste'], 'entreprise': e['entreprise'], 'ville': e['ville'],
             'type_contrat': e['type_contrat'], 'mode': e['mode_travail'], 'langue': e['langue_offre'],
             'competences': ', '.join(e['competences'][:20])}
    for p in profils:
        ligne[f"note_{p['id']}"] = ''
    lignes.append(ligne)

if os.path.exists(NOTATION_REMPLIE):
    print('notation_remplie.csv existe déjà : rien à faire.')
else:
    pd.DataFrame(lignes).to_csv(NOTATION, index=False, encoding='utf-8-sig')
    print(f'{len(lignes)} offres à noter -> {NOTATION}')
    print('1. Ouvre notation.csv avec Google Sheets (double-clic dans Drive).')
    print('2. Remplis les colonnes note_* avec 0, 1, 2 ou 3 (ouvre l’url si besoin).')
    print('3. Fichier -> Télécharger -> CSV, renomme-le notation_remplie.csv et place-le dans', MATCHING)

In [ ]:
# 5. Embeddings multilingues (FR / EN / AR) — environ 1 min sur CPU
from sentence_transformers import SentenceTransformer

encodeur = SentenceTransformer('intfloat/multilingual-e5-base')
_cache = {}

def emb(textes, prefixe='query: '):
    manquants = [t for t in textes if (prefixe, t) not in _cache]
    if manquants:
        vecs = encodeur.encode([prefixe + t for t in manquants], normalize_embeddings=True,
                               batch_size=64, show_progress_bar=False)
        for t, v in zip(manquants, vecs):
            _cache[(prefixe, t)] = v
    return np.array([_cache[(prefixe, t)] for t in textes])

for p in profils:
    p['_comp'] = ensemble(p['competences'])
    p['_doc'] = emb([p['texte'] + ' Compétences : ' + ', '.join(p['competences'])], 'query: ')[0]
print('Profils encodés.')

In [ ]:
# 6. Score de matching + explication
SEUILS = [0.80, 0.84, 0.88]   # seuil de similarité à partir duquel une compétence « proche » rapporte des points

def credit(sim, seuil):
    """Crédit partiel : 0 sous le seuil, 1 à seuil + 0,10, linéaire entre les deux."""
    return float(np.clip((sim - seuil) / 0.10, 0, 1))

def composantes(profil, extraction, texte_offre):
    comp_o = ensemble(extraction.get('competences'))
    comp_p = profil['_comp']
    communes = comp_o & comp_p
    restantes = sorted(comp_o - comp_p)

    # pour chaque compétence non couverte : la compétence du profil la plus proche et sa similarité
    if restantes and comp_p:
        lp = sorted(comp_p)
        sims = emb(restantes) @ emb(lp).T
        plus_proches = [(c, lp[int(sims[i].argmax())], float(sims[i].max())) for i, c in enumerate(restantes)]
    else:
        plus_proches = [(c, None, 0.0) for c in restantes]

    t = extraction.get('type_contrat')
    c_type = 0.5 if t is None else float(t in profil['type_recherche'])
    v, mode = extraction.get('ville'), extraction.get('mode_travail')
    if mode == 'remote' and profil.get('accepte_remote'):
        c_ville = 1.0
    else:
        c_ville = 0.5 if v is None else float(cle(v) in {cle(x) for x in profil['villes']})
    c_langue = float(extraction.get('langue_offre') in profil['langues'])

    return {
        'exacte': len(communes) / len(comp_o) if comp_o else 0.0,
        'doc': float(emb([texte_offre[:2000]], 'passage: ')[0] @ profil['_doc']),
        'pratique': (c_type + c_ville + c_langue) / 3,
        '_communes': sorted(communes), '_plus_proches': plus_proches, '_n_offre': len(comp_o),
    }

def semantique(c, seuil):
    credits = [1.0] * len(c['_communes']) + [credit(sim, seuil) for _, _, sim in c['_plus_proches']]
    return float(np.mean(credits)) if credits else 0.0

POIDS_DEFAUT = {'exacte': 0.3, 'semantique': 0.3, 'doc': 0.2, 'pratique': 0.2}
SEUIL_DEFAUT = 0.84

def score(c, poids=POIDS_DEFAUT, seuil=SEUIL_DEFAUT):
    # doc est une similarité e5 (souvent entre 0,7 et 0,9) : on la recentre sur [0, 1]
    doc01 = float(np.clip((c['doc'] - 0.70) / 0.20, 0, 1))
    vals = {'exacte': c['exacte'], 'semantique': semantique(c, seuil), 'doc': doc01, 'pratique': c['pratique']}
    return 100 * sum(poids[k] * vals[k] for k in poids) / sum(poids.values())

def expliquer(c, seuil=SEUIL_DEFAUT):
    proches = [(a, b) for a, b, sim in c['_plus_proches'] if credit(sim, seuil) > 0]
    manquantes = [a for a, b, sim in c['_plus_proches'] if credit(sim, seuil) == 0]
    lignes = [f"  ✅ communes ({len(c['_communes'])}/{c['_n_offre']}) : {', '.join(c['_communes'][:12])}"]
    if proches:
        lignes.append('  🟡 proches : ' + ', '.join(f'{a} ≈ {b}' for a, b in proches[:8]))
    if manquantes:
        lignes.append('  ❌ manquantes : ' + ', '.join(manquantes[:10]))
    return '\n'.join(lignes)

# Calibration : la similarité e5 est élevée même entre mots sans rapport, d'où les seuils.
for a, b in [('React', 'Vue.js'), ('Machine Learning', 'Deep Learning'), ('PostgreSQL', 'MySQL'),
             ('Docker', 'Kubernetes'), ('React', 'Photoshop'), ('Python', 'comptabilité')]:
    print(f'{a:18} ~ {b:15} : {float(emb([cle(a)])[0] @ emb([cle(b)])[0]):.3f}')

# calcul pour toutes les paires (étiquettes manuelles)
COMP = {(p['id'], url): composantes(p, o['extraction'], o['offre']) for p in profils for url, o in offres.items()}
print(len(COMP), 'paires calculées')

In [ ]:
# 7. Les 5 meilleures offres pour chaque profil (sur toutes les offres)
for p in profils:
    classement = sorted(offres, key=lambda u: -score(COMP[(p['id'], u)]))
    print('=' * 90, f"\nPROFIL {p['id']} — {p['description']}")
    for u in classement[:5]:
        e, c = offres[u]['extraction'], COMP[(p['id'], u)]
        print(f"\n{score(c):5.1f}  {e['poste'][:60]} | {e['entreprise']} | {e['type_contrat']} | {e['ville']}")
        print(expliquer(c))

## Évaluation sur tes notes

Trois métriques :
- **NDCG@5** : les 5 offres classées en tête sont-elles celles que tu as le mieux notées ? (1 = classement parfait)
- **Spearman** : corrélation entre le score et ta note, sur toutes les offres.
- **Précision@5** : part des 5 premières offres notées 2 ou 3.

Méthodes comparées : aléatoire, chacune des composantes seule, le score par défaut, et le score avec **poids réglés par validation croisée** (réglés sur une moitié des offres, mesurés sur l'autre, puis inversement).

In [ ]:
# 8. Évaluation
import warnings
from scipy.stats import spearmanr
warnings.filterwarnings('ignore', category=RuntimeWarning)
try:
    from scipy.stats import ConstantInputWarning
    warnings.filterwarnings('ignore', category=ConstantInputWarning)
except ImportError:
    pass

assert os.path.exists(NOTATION_REMPLIE), 'Note d’abord les offres (cellule 4), puis place notation_remplie.csv dans le dossier matching.'
notes = pd.read_csv(NOTATION_REMPLIE, encoding='utf-8-sig')
for p in profils:
    col = f"note_{p['id']}"
    assert notes[col].notna().all(), f'Il manque des notes dans {col}'
URLS = list(notes['url'])

def ndcg_k(scores, gains, k=5):
    ordre = np.argsort(-np.asarray(scores))[:k]
    dcg = sum((2 ** gains[i] - 1) / np.log2(r + 2) for r, i in enumerate(ordre))
    ideal = sorted(gains, reverse=True)[:k]
    idcg = sum((2 ** g - 1) / np.log2(r + 2) for r, g in enumerate(ideal))
    return dcg / idcg if idcg else 0.0

def evaluer(fonction_score, urls, table=COMP):
    res = []
    for p in profils:
        gains = notes.set_index('url').loc[urls, f"note_{p['id']}"].astype(float).values
        s = [fonction_score(table[(p['id'], u)]) for u in urls]
        top = np.argsort(-np.asarray(s))[:5]
        rho = spearmanr(s, gains).statistic if np.std(s) > 0 and np.std(gains) > 0 else np.nan
        res.append({'NDCG@5': ndcg_k(s, gains), 'Spearman': rho,
                    'Précision@5': float(np.mean(gains[top] >= 2))})
    return pd.DataFrame(res).mean()

def aleatoire(urls, n=200):
    rng = np.random.default_rng(0)
    return pd.concat([evaluer(lambda c: rng.random(), urls) for _ in range(n)], axis=1).mean(axis=1)

# grille : poids (pas de 0,1) × seuil de similarité
GRILLE = [(dict(zip(['exacte', 'semantique', 'doc', 'pratique'], w)), seuil)
          for w in itertools.product(np.arange(0, 1.01, 0.1), repeat=4) if abs(sum(w) - 1) < 1e-6
          for seuil in SEUILS]

def regler(urls, table=COMP):
    return max(GRILLE, key=lambda g: evaluer(lambda c: score(c, *g), urls, table)['NDCG@5'])

def validation_croisee(table=COMP, n_tirages=5):
    """Règle les poids sur une moitié, mesure sur l'autre, dans les deux sens, sur plusieurs tirages."""
    mesures, poids_choisis = [], []
    for graine in range(n_tirages):
        u = URLS[:]
        random.Random(graine).shuffle(u)
        a, b = u[:len(u) // 2], u[len(u) // 2:]
        for reg, mes in [(a, b), (b, a)]:
            w, seuil = regler(reg, table)
            poids_choisis.append({**w, 'seuil': seuil})
            mesures.append(evaluer(lambda c: score(c, w, seuil), mes, table))
    return pd.concat(mesures, axis=1).mean(axis=1), pd.DataFrame(poids_choisis).mean()

methodes = {
    'aléatoire': aleatoire(URLS),
    'couverture exacte seule': evaluer(lambda c: c['exacte'], URLS),
    'couverture sémantique seule': evaluer(lambda c: semantique(c, SEUIL_DEFAUT), URLS),
    'similarité texte seule': evaluer(lambda c: c['doc'], URLS),
    'critères pratiques seuls': evaluer(lambda c: c['pratique'], URLS),
    'hybride (poids par défaut)': evaluer(score, URLS),
}
res_cv, poids_moyens = validation_croisee()
methodes['hybride (poids réglés, validation croisée)'] = res_cv

tableau = pd.DataFrame(methodes).T.round(3)
tableau.to_csv(f'{RESULTATS}/matching_evaluation.csv')
print(f'{len(URLS)} offres × {len(profils)} profils')
print('Poids moyens choisis :', poids_moyens.round(2).to_dict())
tableau

## De bout en bout : compétences extraites par le modèle v3

Jusqu'ici, les offres utilisaient tes **étiquettes manuelles**. En production, elles viendront du **modèle v3**. On refait la même évaluation avec les sorties du modèle (déjà enregistrées par les notebooks 05 et 06, aucun GPU nécessaire). L'écart mesure l'impact des erreurs d'extraction sur le matching.

In [ ]:
# 9. Évaluation de bout en bout
def nettoyer(t):
    return str(t).strip().removeprefix('```json').removesuffix('```').strip()

def reparer(brut):
    t = nettoyer(brut)
    i = t.find('"competences"')
    if i < 0:
        return t
    debut = t.index('[', i)
    items, vus = [], set()
    for m in re.finditer(r'"((?:[^"\\]|\\.)*)"', t[debut:]):
        try:
            item = json.loads('"' + m.group(1) + '"')
        except json.JSONDecodeError:
            break
        if item.strip().lower() in vus:
            break
        vus.add(item.strip().lower())
        items.append(item)
    return t[:debut] + json.dumps(items, ensure_ascii=False) + '}'

def lire_pred(brut):
    for f in (nettoyer, reparer):
        try:
            p = json.loads(f(brut))
            if isinstance(p, dict):
                return p
        except Exception:
            pass
    return {}

predictions = {}
for fichier in ['details_finetune_v3_repare.csv', 'details_vierge_v3_gardefous.csv']:
    chemin = f'{RESULTATS}/{fichier}'
    if os.path.exists(chemin):
        for _, r in pd.read_csv(chemin).iterrows():
            predictions[r['url']] = lire_pred(r['sortie_brute'])
    else:
        print('⚠️ introuvable :', chemin)

manquantes = [u for u in URLS if u not in predictions]
print(len(predictions), 'prédictions chargées ;', len(manquantes), 'offres notées sans prédiction')

if not manquantes:
    COMP_V3 = {(p['id'], u): composantes(p, predictions[u], offres[u]['offre']) for p in profils for u in URLS}
    res_cv_v3, _ = validation_croisee(COMP_V3)
    bout_en_bout = pd.DataFrame({
        'étiquettes manuelles': methodes['hybride (poids réglés, validation croisée)'],
        'extraction modèle v3': res_cv_v3,
    }).T.round(3)
    bout_en_bout.to_csv(f'{RESULTATS}/matching_bout_en_bout.csv')
    display(bout_en_bout)

In [ ]:
# 10. Les erreurs du matching : offres où le score et ta note divergent le plus
pm = dict(poids_moyens)
seuil = pm.pop('seuil')
w = pm
for p in profils:
    col = f"note_{p['id']}"
    d = notes[['url', 'poste', col]].copy()
    d['score'] = [score(COMP[(p['id'], u)], w, seuil) for u in d['url']]
    d['rang_score'] = d['score'].rank(ascending=False)
    d['rang_note'] = d[col].rank(ascending=False)
    d['ecart'] = (d['rang_score'] - d['rang_note']).abs()
    print('=' * 80, f"\nPROFIL {p['id']}")
    for _, r in d.sort_values('ecart', ascending=False).head(4).iterrows():
        print(f"\nnote {int(r[col])} | score {r['score']:.1f} | {r['poste'][:60]}")
        print(expliquer(COMP[(p['id'], r['url'])], seuil))